In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

# ============================================================================
# 1. 参数设置
# ============================================================================

USE_LOG_RNA = True
USE_LOG_ATAC = True

CORR_METHOD = "pearson"        # "pearson" 或 "spearman"
CLIP_QUANTILE = None           # 例如 (0.01, 0.99)，不裁剪设为 None

FILTER_EXPRESSED = False
FILTER_MODE = "either"         # "either" 或 "both"
MIN_RNA_VALUE = 0.0
MIN_ATAC_VALUE = 0.0

PEAK_AGG = "sum"               # "sum", "mean", "max"

OUTDIR = Path("./metacell_rna_atac_gene_peak_scatter")
OUTDIR.mkdir(parents=True, exist_ok=True)

algo_file_map = {
    "Supercell": {
        "rna_h5ad": "/workspace/garq/results/5_supercell/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "atac_h5ad": "/workspace/garq/results/5_supercell/three/GSE158013/GSE158013_atac_metacells.h5ad",
    },
    "SEACell": {
        "rna_h5ad": "/workspace/garq/results/3_seacell/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "atac_h5ad": "/workspace/garq/results/3_seacell/three/GSE158013/GSE158013_atac_metacells.h5ad",
    },
    "MetaCellV2": {
        "rna_h5ad": "/workspace/garq/results/4_mc2/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "atac_h5ad": "/workspace/garq/results/4_mc2/three/GSE158013/GSE158013_atac_metacells.h5ad",
    },
}

genes = ["GZMB", "CD8A"]

GENE_REGIONS = {
    "GZMB": {
        "chrom": "chr17",
        "start": 1120285,
        "end": 1120785,
        "window": 0,
    },
    "CD8A": {
        "chrom": "chr2",
        "start": 86786482,
        "end": 86786982,
        "window": 0,
    },
}
CELLTYPE_CANDIDATES = [
    "celltype",
    "cell_type",
    "major_cell_type",
    "leiden",
    "cluster",
]

color_map = {
    "T.CD4.Naive": "#FF6B6B",
    "T.CD4.Memory": "#FF9999",
    "T.CD8.Naive": "#4ECDC4",
    "T.CD8.Effector": "#2C7A7B",
    "T.DoubleNegative": "#A8DADC",
    "B.Naive": "#FFD93D",
    "B.Activated": "#FFA500",
    "Mono.CD14": "#9D4EDD",
    "Mono.CD16": "#C77DFF",
    "NK": "#06D6A0",
    "DC.Myeloid": "#17becf",
    "Platelets": "#95A5A6",
}


# ============================================================================
# 2. 工具函数
# ============================================================================

def find_feature_name(adata, feature_name):
    if feature_name in adata.var_names:
        return feature_name

    lower_map = {str(v).lower(): str(v) for v in adata.var_names}
    key = feature_name.lower()

    if key in lower_map:
        return lower_map[key]

    raise KeyError(
        f"找不到基因: {feature_name}\n"
        f"var_names 前20个: {adata.var_names[:20].tolist()}"
    )


def get_gene_expression(rna, gene):
    gene_used = find_feature_name(rna, gene)
    x = rna[:, gene_used].X

    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    else:
        x = np.asarray(x).flatten()

    return x.astype(float), gene_used


def parse_peak_name(peak):
    parts = str(peak).replace(":", "-").split("-")

    if len(parts) < 3:
        return None, None, None

    chrom = parts[0]

    try:
        start = int(parts[1])
        end = int(parts[2])
    except ValueError:
        return None, None, None

    return chrom, start, end


def find_gene_associated_peaks_by_region(atac, gene):
    if gene not in GENE_REGIONS:
        raise ValueError(f"GENE_REGIONS 里没有 {gene} 的坐标信息")

    region = GENE_REGIONS[gene]
    chrom = region["chrom"]
    start = region["start"] - region["window"]
    end = region["end"] + region["window"]

    matched_peaks = []

    for peak in atac.var_names:
        peak_chrom, peak_start, peak_end = parse_peak_name(peak)

        if peak_chrom is None:
            continue

        overlap = (
            peak_chrom == chrom
            and peak_end >= start
            and peak_start <= end
        )

        if overlap:
            matched_peaks.append(peak)

    if len(matched_peaks) == 0:
        raise ValueError(
            f"没有找到 {gene} 附近的 peak: "
            f"{chrom}:{start}-{end}"
        )

    return matched_peaks, f"{chrom}:{start}-{end}"


def aggregate_peak_accessibility(atac, peaks):
    x = atac[:, peaks].X

    if hasattr(x, "toarray"):
        x = x.toarray()
    else:
        x = np.asarray(x)

    if PEAK_AGG == "sum":
        y = np.asarray(x.sum(axis=1)).flatten()
    elif PEAK_AGG == "mean":
        y = np.asarray(x.mean(axis=1)).flatten()
    elif PEAK_AGG == "max":
        y = np.asarray(x.max(axis=1)).flatten()
    else:
        raise ValueError(f"未知 PEAK_AGG: {PEAK_AGG}")

    return y.astype(float)


def align_metacells(rna, atac):
    common = rna.obs_names.intersection(atac.obs_names)

    if len(common) > 0:
        rna = rna[common, :].copy()
        atac = atac[common, :].copy()
        print(f"Aligned by obs_names: {len(common)} metacells")
    else:
        if rna.n_obs != atac.n_obs:
            raise ValueError(
                f"RNA 和 ATAC metacell 数量不一致，且 obs_names 没有交集: "
                f"RNA={rna.n_obs}, ATAC={atac.n_obs}"
            )
        print("WARNING: RNA/ATAC obs_names 没有交集，但数量相同，假设顺序一致。")

    return rna, atac


def get_cell_types(adata):
    for col in CELLTYPE_CANDIDATES:
        if col in adata.obs.columns:
            return adata.obs[col].astype(str).values, col
    return None, None


def add_unknown_colors(cell_types):
    if cell_types is None:
        return

    for ct in np.unique(cell_types):
        if ct not in color_map:
            rng = np.random.default_rng(abs(hash(ct)) % (2**32))
            color_map[ct] = "#{:02x}{:02x}{:02x}".format(
                rng.integers(0, 256),
                rng.integers(0, 256),
                rng.integers(0, 256),
            )


def transform_values(values, use_log):
    values = np.asarray(values, dtype=float)

    if use_log:
        values = np.log1p(np.maximum(values, 0))
        label = "log1p"
    else:
        label = "raw"

    return values, label


def prepare_pair_values(rna_raw, atac_raw):
    if len(rna_raw) != len(atac_raw):
        raise ValueError(
            f"RNA 和 ATAC metacell 数量不一致: RNA={len(rna_raw)}, ATAC={len(atac_raw)}"
        )

    rna_raw = np.asarray(rna_raw, dtype=float)
    atac_raw = np.asarray(atac_raw, dtype=float)

    valid_mask = np.isfinite(rna_raw) & np.isfinite(atac_raw)
    kept_mask = valid_mask.copy()

    if FILTER_EXPRESSED:
        if FILTER_MODE == "both":
            expressed_mask = (rna_raw > MIN_RNA_VALUE) & (atac_raw > MIN_ATAC_VALUE)
        elif FILTER_MODE == "either":
            expressed_mask = (rna_raw > MIN_RNA_VALUE) | (atac_raw > MIN_ATAC_VALUE)
        else:
            raise ValueError(f"未知 FILTER_MODE: {FILTER_MODE}")

        kept_mask = valid_mask & expressed_mask

    print(
        f"Filter kept {kept_mask.sum()} / {len(kept_mask)} metacells "
        f"(mode={FILTER_MODE}, RNA>{MIN_RNA_VALUE}, ATAC>{MIN_ATAC_VALUE})"
    )

    x = rna_raw[kept_mask]
    y = atac_raw[kept_mask]

    x, rna_label = transform_values(x, USE_LOG_RNA)
    y, atac_label = transform_values(y, USE_LOG_ATAC)

    if CLIP_QUANTILE is not None and len(x) > 0:
        q_low, q_high = CLIP_QUANTILE

        x_low, x_high = np.quantile(x, [q_low, q_high])
        y_low, y_high = np.quantile(y, [q_low, q_high])

        x = np.clip(x, x_low, x_high)
        y = np.clip(y, y_low, y_high)

    return x, y, kept_mask, rna_label, atac_label


def compute_correlation(x, y):
    if len(x) < 3:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    if CORR_METHOD == "pearson":
        return pearsonr(x, y)
    if CORR_METHOD == "spearman":
        return spearmanr(x, y)

    raise ValueError(f"未知 CORR_METHOD: {CORR_METHOD}")


def format_pvalue(p):
    if np.isnan(p):
        return "NA"
    if p < 1e-300:
        return "<1e-300"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.2f}"


# ============================================================================
# 3. 绘图函数
# ============================================================================

def plot_one_gene_for_all_algorithms(gene):
    n_algos = len(algo_file_map)

    fig, axes = plt.subplots(
        1,
        n_algos,
        figsize=(3 * n_algos, 3.35),
        squeeze=False,
    )
    axes = axes[0]

    stats_rows = []

    for ax, (algo_name, paths) in zip(axes, algo_file_map.items()):
        print("=" * 80)
        print(f"Processing: {algo_name} | gene-associated peaks: {gene}")
        print("=" * 80)

        rna = sc.read_h5ad(paths["rna_h5ad"])
        atac = sc.read_h5ad(paths["atac_h5ad"])
        rna, atac = align_metacells(rna, atac)

        rna_expr_raw, gene_used = get_gene_expression(rna, gene)
        peaks, peak_region = find_gene_associated_peaks_by_region(atac, gene)
        atac_access_raw = aggregate_peak_accessibility(atac, peaks)

        print(f"{gene}: found {len(peaks)} peaks in {peak_region}")
        print(f"First peaks: {peaks[:5]}")

        x_plot, y_plot, kept_mask, rna_label, atac_label = prepare_pair_values(
            rna_expr_raw,
            atac_access_raw,
        )

        rho, p_value = compute_correlation(x_plot, y_plot)
        p_text = format_pvalue(p_value)

        cell_types, celltype_col = get_cell_types(rna)
        if cell_types is not None and len(cell_types) == len(kept_mask):
            cell_types = cell_types[kept_mask]
        else:
            cell_types = None

        add_unknown_colors(cell_types)

        if cell_types is not None and len(cell_types) == len(x_plot):
            for ct in np.unique(cell_types):
                mask = cell_types == ct
                ax.scatter(
                    x_plot[mask],
                    y_plot[mask],
                    s=20,
                    alpha=0.55,
                    color=color_map.get(ct, "#999999"),
                    edgecolors="none",
                    rasterized=False,
                )
        else:
            ax.scatter(
                x_plot,
                y_plot,
                s=20,
                alpha=0.55,
                color="#3C5488",
                edgecolors="none",
                rasterized=False,
            )

        ax.set_xlabel(f"{gene_used} RNA ({rna_label})", fontsize=11, fontstyle="italic")
        ax.set_ylabel(
            f"{gene_used}-associated peak accessibility ({atac_label})",
            fontsize=10,
        )

        ax.set_title(
            f"{algo_name}\ncor = {rho:.2f}, p = {p_text}",
            fontsize=10,
            pad=10,
        )

        ax.tick_params(labelsize=9)

        sns.despine(ax=ax)
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)

        stats_rows.append({
            "algorithm": algo_name,
            "gene": gene_used,
            "gene_region": peak_region,
            "n_associated_peaks": len(peaks),
            "peak_agg": PEAK_AGG,
            "n_metacells_before_filter": len(rna_expr_raw),
            "n_metacells_after_filter": len(x_plot),
            "correlation_method": CORR_METHOD,
            "correlation_r": rho,
            "p_value": p_value,
            "p_value_formatted": p_text,
            "use_log_rna": USE_LOG_RNA,
            "use_log_atac": USE_LOG_ATAC,
            "clip_quantile": str(CLIP_QUANTILE),
            "filter_expressed": FILTER_EXPRESSED,
            "filter_mode": FILTER_MODE,
            "min_rna_value": MIN_RNA_VALUE,
            "min_atac_value": MIN_ATAC_VALUE,
            "celltype_col": celltype_col,
            "rna_h5ad": paths["rna_h5ad"],
            "atac_h5ad": paths["atac_h5ad"],
            "associated_peaks": ";".join(peaks),
        })

        print(f"{algo_name}: n={len(x_plot)}, cor={rho:.2f}, p={p_text}")

    plt.tight_layout()

    base_name = f"{gene}_RNA_ATAC_gene_associated_peaks_{CORR_METHOD}"
    pdf_path = OUTDIR / f"{base_name}.pdf"
    png_path = OUTDIR / f"{base_name}.png"

    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")

    return stats_rows


# ============================================================================
# 4. 主程序
# ============================================================================

all_stats = []

for gene in genes:
    all_stats.extend(plot_one_gene_for_all_algorithms(gene))

stats_df = pd.DataFrame(all_stats)
stats_path = OUTDIR / "metacell_rna_atac_gene_peak_correlations_with_pvalue.csv"
stats_df.to_csv(stats_path, index=False)

print("\nCorrelation summary:")
print(stats_df)
print(f"\nSaved stats: {stats_path}")
print("\nDone.")

In [ ]:
import scanpy as sc

for path in [
    "/workspace/garq/results/5_supercell/three/GSE158013/GSE158013_atac_metacells.h5ad",
    "/workspace/garq/results/3_seacell/three/GSE158013/GSE158013_atac_metacells.h5ad",
    "/workspace/garq/results/4_mc2/three/GSE158013/GSE158013_atac_metacells.h5ad",
]:
    atac = sc.read_h5ad(path)
    print("\n", path)
    print(atac)
    print("var columns:", atac.var.columns.tolist())
    print("first peaks:", atac.var_names[:10].tolist())